In [ ]:
## Test for RadDINO 


In [2]:
import pandas as pd
import numpy as np
from mammo_prep import artifacts,io,normalize,viz,windowing
from pathlib import Path


In [3]:
# ROUTES

ANNOTATIONS_CSV = Path(
"/home/enric/Datasets/Original/vindr/finding_annotations.csv"
)

In [23]:
# Import dataset

df =pd.read_csv(ANNOTATIONS_CSV)

df.head(5)
df["healthy"]=["no_lesion" if x == "['No Finding']" else "has_lesion" for x in df["finding_categories"]]


df.head(5)

freq = pd.crosstab(df["healthy"],df["split"])
freq

split,test,training
healthy,,
has_lesion,452,1802
no_lesion,3643,14589


In [24]:
percentage = freq.div(freq.sum(axis=1), axis=0)
percentage

split,test,training
healthy,,
has_lesion,0.200532,0.799468
no_lesion,0.199814,0.800186


In [ ]:
from PIL import Image
from torchvision import transforms
import timm, torch
import pydicom

data_dicom = pydicom.dcmread("/home/enric/Datasets/Original/vindr/images/5a286b78f7e0e3cfe6c861e23a9fa8c3/32a6cb78f0e4a0bdd95c56c4cb6ac673.dicom")
pixels = data_dicom.pixel_array.astype(float)
pixels_rescaling = (pixels-np.min(pixels))/(np.max(pixels)-np.min(pixels))*255.0
image_8bit = np.uint8(pixels_rescaling)

pil_image = Image.fromarray(image_8bit).convert("RGB")

# Load pretrained model from Hugging Face Hub
model = timm.create_model("hf_hub:Snarcy/RadioDino-s16", pretrained=True)
model.eval().to("cuda" if torch.cuda.is_available() else "cpu")

# Prepare image
image = pil_image
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
])
input_tensor = transform(image).unsqueeze(0).to("cuda" if torch.cuda.is_available() else "cpu")

# Extract features
with torch.no_grad():
    embedding = model(input_tensor)

KeyboardInterrupt: 

In [7]:
embedding 

tensor([[-1.7960e+00,  1.5148e-01,  1.5923e-01, -9.7486e-01, -4.3515e-01,
          9.7774e-01, -1.9227e-01,  2.2123e-02,  6.5971e-01,  1.7615e+00,
          1.3768e+00, -8.1276e-01, -2.5027e+00, -8.8963e-01,  2.1723e-01,
          5.4267e-01, -1.1421e-02, -5.2607e-01, -1.9109e+00, -9.6790e-01,
          5.8625e-02, -6.0632e-01, -3.5641e-01,  1.3698e+00, -1.4118e-01,
          1.2773e+00, -1.8319e+00, -1.4355e-01, -4.4146e-01, -3.9980e-01,
          2.0936e+00,  3.1946e-01,  1.0536e+00,  1.2460e+00, -1.5803e+00,
          1.0260e+00, -2.0063e-01, -1.1621e+00, -1.6088e-01, -3.3426e-01,
          7.5673e-02,  1.7306e+00,  7.6373e-01, -2.1101e-01, -3.6847e-01,
          1.0891e+00, -5.1715e-01, -2.9272e-01,  7.5577e-01, -6.4187e-01,
          4.7161e-01,  5.2764e-01,  5.9118e-01,  1.2603e-01, -5.4806e+00,
          2.1030e+00, -5.5163e-01, -5.1547e-02, -7.7596e-01, -5.4223e-01,
          6.0643e-01, -5.0241e-01,  5.5342e-01, -8.1139e-01,  2.2908e-01,
          4.5721e-01, -9.1576e-01,  1.